# Scoring Scorio Lite with `scorio.eval`

This notebook evaluates `gpt-oss-20b_medium` on AIME 2026. Scorio expects an
`M x N` outcome matrix: questions by sampled attempts. Here that is 30 questions by
80 seeds.

|  |  |
| --- | --- |
| module | [`scorio/eval`](https://github.com/mohsenhariri/scorio/tree/main/scorio/eval) |
| method reference | [`scorio/eval/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/README.md) |
| dataset | [Scorio Lite](https://huggingface.co/datasets/harimo/scorio-lite) |
| install | `pip install scorio` |

In [2]:
import pandas as pd
from datasets import load_dataset
from IPython.display import display

from scorio import eval

repo_name = "harimo/scorio-lite"
model_name = "gpt-oss-20b_medium"
task = "aime_2026"

meta = load_dataset(repo_name, "meta-math", split=task)
rows = (meta
        .filter(lambda row: row["model_key"] == model_name)
        .select_columns(["data_id", "seed", "evalscope_is_correct"])
        .to_pandas()
        .sort_values(["data_id", "seed"]))

R = rows.evalscope_is_correct.to_numpy().astype(int).reshape(30, 80)

print(R.shape, R.dtype)
print(R[:3, :12])

Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

(30, 80) int64
[[1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 0 1 1]]


## Bayes@N

`bayes` shrinks question-level rates away from exact zero and one. The `_ci` form also returns a 95% credible interval.

In [3]:
print("Bayes@N ", eval.bayes(R))

mu, sigma, lo, hi = eval.bayes_ci(R)
print(f"95% interval: {mu:.3f} +- {sigma:.3f}  [{lo:.3f}, {hi:.3f}]")

Bayes@N  (0.7699186991869919, 0.006483216659361282)
95% interval: 0.770 +- 0.006  [0.757, 0.783]


## Pass@k, Maj@k, and Pass^k

These metrics answer different questions. `pass_at_k` asks whether at least one of k
samples is correct, `maj_at_k` asks whether a strict majority is correct, and
`pass_hat_k` asks whether all k samples are correct.

In [4]:
ks = [1, 2, 4, 8, 16, 80]
table = pd.DataFrame({
    "pass@k": [eval.pass_at_k(R, k) for k in ks],
    "maj@k": [eval.maj_at_k(R, k) for k in ks],
    "pass^k": [eval.pass_hat_k(R, k) for k in ks],
    "auc@k": [eval.auc_at_k(R, k) for k in ks],
}, index=pd.Index(ks, name="k"))

display(table.round(3))

,pass@k,maj@k,pass^k,auc@k
k,,,,
1,0.777,0.777,0.777,0.777
2,0.875,0.678,0.678,0.826
4,0.919,0.765,0.558,0.876
8,0.939,0.814,0.433,0.907
16,0.956,0.849,0.312,0.929
80,0.967,0.900,0.067,0.959


## Effect of sample count

Because rows are ordered by seed, slicing the first n columns gives a reproducible budget
sweep. Each larger budget adds later seeds to the same prefix. On this split, the estimate
changes with the prefix while the interval steadily narrows.

In [5]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame(
    [eval.bayes_ci(R[:, :n]) for n in budgets],
    columns=["mu", "sigma", "lo", "hi"],
    index=pd.Index(budgets, name="samples"),
)
sweep["width"] = sweep.hi - sweep.lo

display(sweep.round(3))

,mu,sigma,lo,hi,width
samples,,,,,
1,0.622,0.043,0.538,0.707,0.169
2,0.650,0.037,0.578,0.722,0.143
4,0.694,0.028,0.640,0.749,0.109
8,0.740,0.020,0.701,0.779,0.078
16,0.756,0.014,0.728,0.784,0.056
32,0.774,0.010,0.753,0.794,0.040
80,0.770,0.006,0.757,0.783,0.025


## Comparing all four configurations

In [6]:
all_rows = (meta
            .select_columns(["model_key", "data_id", "seed", "evalscope_is_correct"])
            .to_pandas()
            .sort_values(["model_key", "data_id", "seed"]))

models = sorted(all_rows.model_key.unique())
R_all = all_rows.evalscope_is_correct.to_numpy().astype(int).reshape(len(models), 30, 80)

comparison = pd.DataFrame(
    [eval.bayes_ci(R_all[i]) for i in range(len(models))],
    columns=["mu", "sigma", "lo", "hi"],
    index=models,
)

display(comparison.sort_values("mu", ascending=False).round(3))

,mu,sigma,lo,hi
Qwen3.6-35B-A3B,0.912,0.004,0.904,0.920
gpt-oss-20b_high,0.867,0.005,0.857,0.877
gpt-oss-20b_medium,0.770,0.006,0.757,0.783
gpt-oss-20b_low,0.427,0.007,0.413,0.441


## Related estimators

The module also includes generalized Pass@k, expected-best reward, geometric estimators,
and categorical outcomes. The [method reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/README.md)
lists each estimator and its source.
